# CNN VAE for SRL in Simulation (DonkeySim)

This notebook trains a CNN VAE (beta variant) for State Representation Learning (SRL) using simulation data from DonkeySim. The resulting model compresses camera images into a low-dimensional latent space for use in reinforcement learning (e.g., SAC).

**Key Changes for Simulation:**
- Data collection: Use DonkeySim to collect 1k-10k images (e.g., via simulator scripts). Images are cleaner than real-world, so add optional augmentation if needed.
- Preprocessing: Adjusted for sim resolution (default 160x120 in DonkeySim); keep crop to focus on track.
- Improvements: Added validation split, normalization, early stopping, configurable z_dim/epochs, built-in TensorBoard (no TensorBoardX).

First, collect training data in DonkeySim (e.g., run sim with random policy to capture diverse paths: center, sides, zigzags). Upload ZIP to Google Drive.

Run in Google Colab for Drive integration.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import argparse

# Parse args (for Colab, use defaults or modify here)
parser = argparse.ArgumentParser()
parser.add_argument('--data_file', default='', help='Dataset ZIP name (e.g., sim_data.zip)')
parser.add_argument('--data_dir', default='dataset', help='Unzipped dataset folder')
parser.add_argument('--z_dim', type=int, default=32, help='Latent dimension')
parser.add_argument('--epochs', type=int, default=100, help='Training epochs')
parser.add_argument('--batch_size', type=int, default=64, help='Batch size')
parser.add_argument('--patience', type=int, default=10, help='Early stopping patience')
args = parser.parse_args([])  # For Colab; use parser.parse_args() in script mode

DATASET_FILE = args.data_file
DATASET_DIR = args.data_dir
DATASET_ZIP = os.path.join(DATASET_DIR, DATASET_FILE)

## Copy from Google Drive

Copy simulation training data ZIP from Drive and unzip. Assume data collected from DonkeySim (e.g., generated tracks).

In [ ]:
!rm -rf dataset_root
!cp "/content/drive/My Drive/{DATASET_ZIP}" ./
!unzip -q {DATASET_FILE}

!mkdir dataset_root
!mv {DATASET_DIR} './dataset_root'

## Import Modules

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets
from torchvision import transforms
from torchvision.utils import save_image
from IPython.display import Image
from IPython.core.display import Image, display
from torch.utils.data import random_split
import numpy as np
from torchsummary import summary

%load_ext autoreload
%autoreload 2
%load_ext tensorboard

## Load GPU Device

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Load Dataset (with Validation Split)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((120, 160)),
    transforms.Lambda(lambda x: x.crop((0, 40, 160, 120))),  # Adjust if sim FOV different
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),  # Added for stability
    # Optional: Add augmentation for sim data if too clean: transforms.ColorJitter(brightness=0.2, contrast=0.2)
])
dataset = datasets.ImageFolder(root='./dataset_root', transform=transform)
if len(dataset) == 0:
    raise ValueError("Dataset empty! Check data collection from DonkeySim.")

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=args.batch_size, shuffle=False, num_workers=2, pin_memory=True)
print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

In [ ]:
fixed_x, _ = next(iter(dataloader))
save_image(fixed_x, 'real_image.png')
Image('real_image.png')

## Define VAE Network (with Beta Param)

In [ ]:
class Flatten(nn.Module):
    def forward(self, input):
        return input.view(input.size(0), -1)

class UnFlatten(nn.Module):
    def forward(self, input, size=256):
        return input.view(input.size(0), size, 3, 8)


class VAE(nn.Module):
    def __init__(self, image_channels=3, h_dim=6144, z_dim=32, beta=5.0):
        super(VAE, self).__init__()
        self.z_dim = z_dim
        self.beta = beta
        self.encoder = nn.Sequential(
            nn.Conv2d(image_channels, 32, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size=4, stride=2),
            nn.ReLU(),
            Flatten()
        )

        self.fc1 = nn.Linear(h_dim, z_dim)
        self.fc2 = nn.Linear(h_dim, z_dim)
        self.fc3 = nn.Linear(z_dim, h_dim)

        self.decoder = nn.Sequential(
            UnFlatten(),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=5, stride=2),
            nn.ReLU(),
        )

        self.out1 = nn.Sequential(nn.ConvTranspose2d(32, image_channels, kernel_size=4, stride=2),
                                  nn.Sigmoid(),
                                  )
        self.out2 = nn.Sequential(nn.ConvTranspose2d(32, image_channels, kernel_size=4, stride=2),
                                  nn.Sigmoid(),
                                  )

    def reparameterize(self, mu, logvar):
        std = logvar.mul(0.5).exp_()
        esp = torch.randn(*mu.size()).to(device)
        z = mu + std * esp
        return z

    def bottleneck(self, h):
        mu, logvar = self.fc1(h), self.fc2(h)
        z = self.reparameterize(mu, logvar)
        return z, mu, logvar

    def encode(self, x):
        h = self.encoder(x)
        z, mu, logvar = self.bottleneck(h)
        return z, mu, logvar

    def decode(self, z):
        z = self.fc3(z)
        x = self.decoder(z)
        mu_y = self.out1(x)
        sigma_y = self.out2(x)
        return mu_y, sigma_y

    def forward(self, x):
        z, mu, logvar = self.encode(x)
        mu_y, sigma_y = self.decode(z)
        return mu_y, sigma_y, mu, logvar

    def loss_fn(self, image, mu_y, sigma_y, mean, logvar):
        m_vae_loss = (mu_y - image)**2 / sigma_y
        m_vae_loss = 0.5 * torch.sum(m_vae_loss)
        a_vae_loss = torch.log(2.0 * torch.pi * sigma_y)
        a_vae_loss = 0.5 * torch.sum(a_vae_loss)
        KL = -0.5 * torch.sum((1 + logvar - mean.pow(2) - logvar.exp()), dim=0)
        KL = torch.mean(KL)
        return torch.mean((KL * self.beta) + (10 * m_vae_loss) + a_vae_loss)

## Prepare Training

Create VAE model and initialize optimizer.

In [ ]:
image_channels = fixed_x.size(1)
vae = VAE(image_channels=image_channels, z_dim=args.z_dim).to(device)
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)
summary(vae, (3, 80, 160))

## TensorBoard

In [ ]:
%tensorboard --logdir ./runs

## Start Training (with Early Stopping and Val)

In [ ]:
from tensorboardX import SummaryWriter
writer = SummaryWriter()

vae.train()
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(args.epochs):
    losses = []
    for idx, (images, _) in enumerate(dataloader):
        images = images.to(device, non_blocking=True)
        optimizer.zero_grad()
        mu_y, sigma_y, mu, logvar = vae(images)
        loss = vae.loss_fn(images, mu_y, sigma_y, mu, logvar)
        loss.backward()
        optimizer.step()
        losses.append(loss.cpu().detach().numpy())

    avg_train_loss = np.average(losses)
    writer.add_scalar('Loss/train', avg_train_loss, epoch)
    print(f"EPOCH: {epoch+1} train loss: {avg_train_loss}")

    # Validation
    vae.eval()
    val_losses = []
    grid = None
    grid_sigma = None
    with torch.no_grad():
        for images, _ in val_dataloader:
            images = images.to(device)
            mu_y, sigma_y, mu, logvar = vae(images)
            val_loss = vae.loss_fn(images, mu_y, sigma_y, mu, logvar)
            val_losses.append(val_loss.item())
            grid = torchvision.utils.make_grid(mu_y[:16])  # Log sample recons
            grid_sigma = torchvision.utils.make_grid(sigma_y[:16])

    avg_val_loss = np.average(val_losses)
    writer.add_scalar('Loss/val', avg_val_loss, epoch)
    writer.add_image('Image/reconst', grid, epoch)
    writer.add_image('Image/sigma', grid_sigma, epoch)
    print(f"EPOCH: {epoch+1} val loss: {avg_val_loss}")

    # Early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(vae.state_dict(), 'best_vae.torch')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= args.patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

torch.save(vae.state_dict(), 'vae.torch')
writer.close()

## Visualize Latent Space
Visualizing latent space by TensorBoard.
You can visualize latent space with TensorBoard Projector view.
The latent spaces are auto labeled by K-means. If similar images stick together, we consider the quality of the latent space to be good.


In [ ]:
from sklearn.cluster import KMeans
from tensorboardX import SummaryWriter
writer = SummaryWriter()

vae.eval()
latent_spaces = []
for idx, (images, _) in enumerate(dataloader):
    images = images.to(device)
    z, _, _ = vae.encode(images)
    z = z.detach().cpu().numpy()
    latent_spaces.append(z)
    if len(latent_spaces) * args.batch_size > 5000:
        break
latent_spaces = np.concatenate(latent_spaces, axis=0)[:5000]

images_recon, sigma_y = vae.decode(torch.Tensor(latent_spaces).to(device))
images_recon = F.interpolate(images_recon, size=(40, 40), mode='bilinear', align_corners=False)

kmeans_model = KMeans(n_clusters=5, verbose=0, n_init=10)
labels = kmeans_model.fit_predict(latent_spaces)

writer.add_embedding(mat=latent_spaces, metadata=labels, label_img=images_recon)
writer.close()

## Re-launch TensorBoard
Reload the page if projector tab doesn't show.

In [ ]:
%tensorboard --logdir ./runs

## Cleanup

Copy trained model file to GoogleDrive.

In [ ]:
!cp vae.torch '/content/drive/My Drive/vae.torch'
!cp best_vae.torch '/content/drive/My Drive/best_vae.torch'